# 02B | India-Nepal: data understanding, preparation and EDA

Inspect disaster relevance, flood/earthquake type and humanitarian-information tasks separately. Analyze class/source composition, frequent terms, word/token lengths and related-record grouping. Preserve distinct labels, negations and protected splits.

No predictive models are trained here. This notebook processes only India-Nepal.


## Inputs and outputs

Complete notebook 02A. Inputs are this experiment's public acquisition manifests and raw India-Nepal files, restored from Google Drive.

Outputs: Twelve task-specific chart panels; label/source support; actual tokenizer-length coverage; independently validated India-Nepal preparation.


## Configuration

Enable Drive, bootstrap and stage execution. Keep the experiment ID unchanged across the package. All actions default disabled.


In [ ]:
EXPERIMENT_ID = "colab-cpu-v3-clean"
DRIVE_FOLDER = "ChainShield-Colab"
ENABLE_DRIVE = False
BOOTSTRAP_EXPERIMENT = False
ALLOW_PACKAGE_INSTALL = False
ALLOW_DATA_DOWNLOADS = False
ALLOW_PRETRAINED_DOWNLOADS = False
RUN_STAGE = False
SYNC_INTERVAL_SECONDS = 60
VERIFY_PERSISTENCE_FULL = False
VERIFY_CHECKPOINT_PAIRS = False
CLEAR_STALE_WRITER_LOCK = False
CONFIRM_NO_OTHER_WRITER = False
DOWNLOAD_REPORT_ZIP = False

# Isolated plotting workers must not inherit the Colab inline backend.
if RUN_STAGE:
    import os
    os.environ["MPLBACKEND"] = "Agg"


## Shared definitions

Readable code is supplied once by its owning stage and persisted in Drive. Register these definitions before attaching; registration alone performs no preparation.


In [ ]:
# This small cell registers the visible stage code; it writes no files.
REGISTERED_FILES = {}
def register_stage_file(line, body):
    name=line.strip()
    if not name or name.startswith("/") or "\\" in name or ":" in name or ".." in name.split("/"):
        raise ValueError("Use a safe relative module name")
    REGISTERED_FILES[name]=body.rstrip("\n")+"\n"
get_ipython().register_magic_function(register_stage_file,"cell","chainshield_file")


### src/chainshield/data/india_nepal.py


In [ ]:
%%chainshield_file src/chainshield/data/india_nepal.py
"""Task-aware India-Nepal preparation directly from the unmodified ZIP."""
from __future__ import annotations
from collections import Counter, defaultdict
import csv
import hashlib
import html
import io
from pathlib import Path
import re
import unicodedata
import zipfile






def reliable_id(value):
    return bool(re.fullmatch(r"[0-9]{15,22}", str(value).strip()))


def connected_groups(rows):
    """Connect exact text and reliable same-namespace IDs, including transitive links."""
    parent = list(range(len(rows)))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    def union(a, b):
        a, b = find(a), find(b)
        if a != b:
            parent[max(a,b)] = min(a,b)
    seen = {}
    for i, row in enumerate(rows):
        tokens = []
        if row["text_key"]:
            tokens.append("text:" + row["text_key"])
        if reliable_id(row["raw_tweet_id"]):
            tokens.append("id:" + row["collection"] + ":" + row["raw_tweet_id"])
        for token in tokens:
            if token in seen:
                union(i, seen[token])
            else:
                seen[token] = i
    members = defaultdict(list)
    for i, row in enumerate(rows):
        members[find(i)].append(row)
    for group in members.values():
        group_id = hashlib.sha256(min(r["record_id"] for r in group).encode()).hexdigest()
        for row in group:
            row["group_id"] = group_id


def seeded_group_split(group_id, seed, fractions):
    value = int(hashlib.sha256(f"{seed}:{group_id}".encode()).hexdigest(), 16) / (1 << 256)
    return "train" if value < fractions["train"] else (
        "dev" if value < fractions["train"] + fractions["dev"] else "test")


def apply_group_policy(rows, seed, fractions):
    connected_groups(rows)
    tasks = defaultdict(list)
    for row in rows:
        row["split"] = seeded_group_split(row["group_id"], seed, fractions)
        tasks[(row["task"], row["group_id"])].append(row)
    for group in tasks.values():
        labels = {r["label"] for r in group if r["label"]}
        conflict = len(labels) > 1
        eligible = sorted((r for r in group if not r["_reasons"]), key=lambda r: r["record_id"])
        representative = eligible[0]["record_id"] if eligible else None
        for row in group:
            if conflict:
                row["_reasons"].append("conflicting_labels_within_task_group")
            elif not row["_reasons"] and row["record_id"] != representative:
                row["_reasons"].append("redundant_task_group_member")
            row["eligible"] = not row["_reasons"]
            row["exclusion_reasons"] = "|".join(sorted(set(row.pop("_reasons"))))


def prepare(raw_root: Path, config):
    spec = config["india_nepal"]
    rows, source_summary = [], []
    with zipfile.ZipFile(raw_root / "dataset.zip") as archive:
        for source in spec["files"]:
            name = source["path"]
            payload = archive.read(name).decode(spec["encoding"])
            reader = csv.DictReader(io.StringIO(payload))
            required = {"Tweet_id", "Tweet_text", "Target"}
            if source.get("label_field"):
                required.add(source["label_field"])
            if not required.issubset(reader.fieldnames or []):
                raise ValueError(f"Missing source fields: {name}")
            source_ids, source_texts, target_counts = Counter(), Counter(), Counter()
            source_count = 0
            for line, raw in enumerate(reader, 2):
                if None in raw:
                    raise ValueError(f"Malformed CSV row in {name}, line {line}")
                source_count += 1
                tid = str(raw.get("Tweet_id") or "").strip()
                text = clean_text(raw.get("Tweet_text"))
                key = text_key(text)
                reasons = []
                label_field = source.get("label_field", "Target")
                original_label = str(raw.get(label_field) or "").strip()
                mapping = spec["help_labels"] if label_field == "Help" else source["labels"]
                label = mapping.get(original_label, "")
                if not label:
                    reasons.append("non_disaster_outside_type_task" if
                        source["task"] == "disaster_type" and
                        original_label in spec["out_of_scope_disaster_type_values"]
                        else "invalid_or_missing_label")
                if not text:
                    reasons.append("empty_text")
                source_ids[tid] += 1
                source_texts[key] += 1
                target_counts[original_label] += 1
                rows.append({
                    "record_id": f"{Path(name).name}:{line:08d}",
                    "source_file": name, "source_row": line,
                    "collection": source["collection"], "raw_tweet_id": tid,
                    "id_reliable": reliable_id(tid), "task": source["task"],
                    "raw_label": original_label, "label": label,
                    "text": text, "text_key": key, "_reasons": reasons,
                })
            source_summary.append({
                "file": name, "task": source["task"], "rows": source_count,
                "unique_raw_ids": len(source_ids),
                "reliable_id_rows": sum(n for tid,n in source_ids.items() if reliable_id(tid)),
                "scientific_notation_id_rows": sum(n for tid,n in source_ids.items()
                    if re.fullmatch(r"[0-9.]+[Ee][+-]?[0-9]+",tid)),
                "unique_normalized_texts": len(source_texts),
                "raw_label_counts": dict(target_counts),
            })
    apply_group_policy(rows, config["seed"], config["splits"])
    manifest = [{key: r[key] for key in
        ("record_id","task","group_id","text_key","split","label","eligible","exclusion_reasons")}
        for r in rows]
    labels = {
        "disaster_relevance": {"0":"non_disaster","1":"disaster"},
        "disaster_type": {"1":"flood","2":"earthquake"},
        "humanitarian_information": spec["help_labels"],
    }
    quality = {
        "source_rows": len(rows), "canonical_rows": len(rows),
        "sources": source_summary, "connected_groups": len({r["group_id"] for r in rows}),
        "split_policy": spec["split_policy"],
        "exclusion_reason_counts": dict(Counter(reason for r in rows
            for reason in r["exclusion_reasons"].split("|") if reason)),
        "tasks": {},
        "limitations": [
            "Scientific-notation and decimal-form IDs are not reconstructed or used as identity keys.",
            "Original-source IDs and India IDs use separate identity namespaces.",
            "Exact text grouping spans tasks and collections; unreliable IDs rely on text.",
            "Seeded group hashing is deterministic, not stratified; per-class support must be checked.",
            "Near-duplicate, template-family and event-held-out evaluation remain pending.",
            "Do not claim all India records are independently verified historical posts.",
            "Humanitarian labels here differ from CrisisMMD; tasks remain separate.",
        ],
    }
    for task in labels:
        selected = [r for r in rows if r["task"] == task]
        valid = [r for r in selected if r["eligible"]]
        quality["tasks"][task] = {
            "source_rows":len(selected), "eligible_rows":len(valid),
            "excluded_rows":len(selected)-len(valid),
            "eligible_labels":dict(Counter(r["label"] for r in valid)),
            "split_counts":dict(Counter(r["split"] for r in valid)),
            "split_label_counts":{s:dict(Counter(r["label"] for r in valid if r["split"]==s))
                                  for s in ("train","dev","test")},
        }
    return {"canonical":rows,"split_manifest":manifest,"quality":quality,"label_map":labels}

from .text_utils import clean_text,text_key


### configs/preprocessing/india_nepal.json


In [ ]:
%%chainshield_file configs/preprocessing/india_nepal.json
{
  "schema_version": 1,
  "pipeline_version": "chainshield-preparation-v1",
  "seed": 42,
  "splits": {
    "train": 0.7,
    "dev": 0.15,
    "test": 0.15
  },
  "normalization": "HTML unescape, Unicode NFC, whitespace collapse; casefold only for exact-text keys",
  "india_nepal": {
    "encoding": "utf-8-sig",
    "split_policy": "Seeded hash on connected exact-text/reliable-ID groups shared across tasks; not claimed stratified",
    "reliable_id_policy": "Only 15-22 ASCII digits; namespace India vs original collections; scientific notation and decimal IDs retained only as provenance",
    "files": [
      {
        "path": "dataset/TweetMasterData.csv",
        "task": "disaster_relevance",
        "collection": "original",
        "labels": {
          "0": "non_disaster",
          "1": "disaster"
        }
      },
      {
        "path": "dataset/INDIANTWEETMASTER.csv",
        "task": "disaster_relevance",
        "collection": "india",
        "labels": {
          "0": "non_disaster",
          "1": "disaster"
        }
      },
      {
        "path": "dataset/TWEETDATASET( CLASS 0,1 AND 2).csv",
        "task": "disaster_type",
        "collection": "original",
        "labels": {
          "1": "flood",
          "2": "earthquake"
        }
      },
      {
        "path": "dataset/INDIANTWEETDATASET.csv",
        "task": "disaster_type",
        "collection": "india",
        "labels": {
          "1": "flood",
          "2": "earthquake"
        }
      },
      {
        "path": "dataset/NEPALEARTHQUAKE.csv",
        "task": "humanitarian_information",
        "collection": "original",
        "label_field": "Help"
      },
      {
        "path": "dataset/INDIANEARTHQUAKE.csv",
        "task": "humanitarian_information",
        "collection": "india",
        "label_field": "Help"
      },
      {
        "path": "dataset/INDIANFLOOD.csv",
        "task": "humanitarian_information",
        "collection": "india",
        "label_field": "Help"
      }
    ],
    "help_labels": {
      "Other useful information": "other_useful_information",
      "Donations and volunteering": "donations_and_volunteering",
      "Sympathy and support": "sympathy_and_support",
      "Food and Medicine": "food_and_medicine",
      "Infrastructure and utilities": "infrastructure_and_utilities"
    },
    "out_of_scope_disaster_type_values": [
      "0"
    ]
  }
}


### src/chainshield/research/eda_india_nepal.py


In [ ]:
%%chainshield_file src/chainshield/research/eda_india_nepal.py
from .eda import truth,text_profile

def add_profiles(profile,canonical,prepared,tokenizer,include_examples,root,inherited,config):
    india=[r for r in canonical.get("india_nepal",[]) if r["split"]=="train" and truth(r["eligible"])]
    for task in sorted({r["task"] for r in india}):
        records=[{"record_id":r["record_id"],"group_id":r["group_id"],"split":"train","text":r["text"],"label":r["label"],"source":r["collection"]} for r in india if r["task"]==task]
        profile["text_tasks"]["india_nepal/"+task]=text_profile(records,task,tokenizer,include_examples=include_examples)


In [ ]:
import hashlib,json
EXPECTED_STAGE_FILES={'src/chainshield/data/india_nepal.py': 'feb95db2b05b9e201946c06fa78dcd45e30111fa16a23a851198a390dcc83892', 'configs/preprocessing/india_nepal.json': '6e7ed78010ac5baf4772ee0b91c16c6da665a3226ec5908b84d7ae662eecda2a', 'src/chainshield/research/eda_india_nepal.py': '25322d019b033a4a7ab8e91fa24850135317a7f997dc01ec935f22a9e4b56d2c'}
assert set(REGISTERED_FILES)==set(EXPECTED_STAGE_FILES)
for name,expected in EXPECTED_STAGE_FILES.items():
    if hashlib.sha256(REGISTERED_FILES[name].encode()).hexdigest()!=expected:raise ValueError("Changed source: "+name)


## Attach and verify prerequisites


In [ ]:
# Connect the Colab experiment and load the shared runtime.
from pathlib import Path
CONTROLLER_SHA = '8c092c59baffc0804aeca86b4f0ac3fb6d4be3cd96eea96c9e7299e559d9c9e6'
IMPLEMENTATION_SHA256 = CONTROLLER_SHA
rt = None
if BOOTSTRAP_EXPERIMENT:
    if not ENABLE_DRIVE:
        raise PermissionError("Set ENABLE_DRIVE=True")
    from google.colab import drive
    drive.mount("/content/drive")
    runtime_path=Path("/content/drive/MyDrive")/DRIVE_FOLDER/"experiments"/EXPERIMENT_ID/"workspace/.replication/controller.py"
    if not runtime_path.is_file():
        raise RuntimeError("Complete Version 3 notebook 00 first; its shared controller is missing")
    runtime_code=runtime_path.read_text(encoding="utf-8")
    if hashlib.sha256(runtime_code.encode()).hexdigest()!=CONTROLLER_SHA:
        raise ValueError("Shared runtime version differs; do not mix notebook packages")
    exec(compile(runtime_code,"<Colab shared runtime>","exec"),globals())
    rt=Replica(EXPERIMENT_ID,drive_base=DRIVE_FOLDER,allow_drive=True)
    if CLEAR_STALE_WRITER_LOCK:
        rt.clear_stale_lock(acknowledge_no_other_writer=CONFIRM_NO_OTHER_WRITER)
else:
    print("Bootstrap disabled: no Drive access or filesystem changes")


In [ ]:
if RUN_STAGE:
    if rt is None:raise RuntimeError("Enable Drive/bootstrap first")
    rt.require_stage('01')
    rt.install_sources(REGISTERED_FILES)
    rt.environment(allow_install=ALLOW_PACKAGE_INSTALL)
else:
    print("Stage execution disabled")


## Data understanding and preparation choices

Inspect disaster relevance, flood/earthquake type and humanitarian-information tasks separately. Analyze class/source composition, frequent terms, word/token lengths and related-record grouping. Preserve distinct labels, negations and protected splits.

Preparation verifies public input hashes and expected semantic/test-membership digests. All analysis below uses eligible training records; final-test labels never select settings.


In [ ]:
if RUN_STAGE:
    rt.execute("02B-prepare",["scripts/dataset_research_workflow.py","prepare","--dataset","india_nepal"],SYNC_INTERVAL_SECONDS)
else:
    print("Stage execution disabled")


## Pinned tokenizer for measured text lengths

Tokenizer assets are independent of classifier fine-tuning. India-Nepal acquires them once; CrisisMMD reuses the same verified assets. No model weights are downloaded here.


In [ ]:
if RUN_STAGE:
    path=rt.root/"artifacts/research-v2/tokenizer/distilbert/manifest.json"
    if not path.exists():
        if not ALLOW_PRETRAINED_DOWNLOADS:raise PermissionError("Enable pinned tokenizer acquisition with ALLOW_PRETRAINED_DOWNLOADS=True")
        rt.execute("02B-tokenizer",["scripts/acquire_tokenizer_assets.py"],SYNC_INTERVAL_SECONDS)
else:
    print("Stage execution disabled")


## Measured EDA, interpretation and decisions

Read the observation and preparation implication for every chart. Coverage and correlation do not prove a model will perform better.


In [ ]:
if RUN_STAGE:
    rt.execute("02B-eda",["scripts/dataset_research_workflow.py","eda","--dataset","india_nepal"],SYNC_INTERVAL_SECONDS)
else:
    print("Stage execution disabled")


In [ ]:
if RUN_STAGE:
    print(json.dumps(read_json(rt.root/"data/processed/"/"india_nepal"/"quality.json"),indent=2))
else:
    print("Stage execution disabled")


In [ ]:
if RUN_STAGE:
    from IPython.display import display,Image,Markdown
    for variant in ('control',):
        folder=rt.root/"artifacts/research-v2/eda"/("india_nepal_"+variant)
        display(Markdown("### India-Nepal | "+variant))
        display(Markdown((folder/"EDA_Review.md").read_text()))
        for image in sorted((folder/"figures_v2").glob("*.png")):display(Image(filename=str(image)))
        summary=read_json(folder/"dataset-profiles.json")
        print("Preparation safeguards:",json.dumps(summary["preparation_controls"],indent=2))
else:
    print("Stage execution disabled")


## Persist and complete

Finish only after preparation and charts succeed. A restart of the Python kernel alone may retain partial scratch files; use a clean temporary Colab runtime for archive restoration when needed. Preserve Drive evidence.


In [ ]:
if RUN_STAGE:
    rt.finish_stage("02B")
else:
    print("Stage execution disabled")
